In [4]:
import pandas as pd
import time
import sys
from pathlib import Path
import json


In [5]:

from kafka import KafkaProducer


url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = [ \
    'lpep_pickup_datetime', \
    'lpep_dropoff_datetime', \
    'PULocationID', \
    'DOLocationID', \
    'passenger_count', \
    'trip_distance', \
    'tip_amount', \
    'total_amount' \
    ]

df = pd.read_parquet(url, columns=columns)
df = df.fillna(0)

In [6]:
import json
import dataclasses
from dataclasses import dataclass


@dataclass
class Ride:
    lpep_pickup_datetime: str  
    lpep_dropoff_datetime: str 
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float


def ride_from_row(row):
    return Ride(
        lpep_pickup_datetime=str(row['lpep_pickup_datetime']),
        lpep_dropoff_datetime=str(row['lpep_dropoff_datetime']),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=int(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount'])
    )


def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    ride_json_str = json.dumps(ride_dict).encode('utf-8')
    return ride_json_str


def ride_deserializer(ride_json_str):
    ride_dict = json.loads(ride_json_str.decode('utf-8'))
    ride = Ride(**ride_dict)
    return ride

In [7]:
df.head(5)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [8]:
ride = ride_from_row(df.iloc[0])

In [9]:
ride

Ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [10]:
ride_serializer(ride)

b'{"lpep_pickup_datetime": "2025-10-01 00:21:47", "lpep_dropoff_datetime": "2025-10-01 00:24:37", "PULocationID": 247, "DOLocationID": 69, "passenger_count": 1, "trip_distance": 0.7, "tip_amount": 1.7, "total_amount": 10.0}'

In [11]:
ride_deserializer(ride_serializer(ride))

Ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [12]:
import re

# 1. Simulate how your current 'ride_from_row' converts timestamps to strings
formatted_dates = df['lpep_pickup_datetime'].apply(lambda x: str(x))

# 2. Define the exact regex for 'yyyy-MM-dd HH:mm:ss'
# This pattern strictly requires: 4 digits, dash, 2 digits, dash, 2 digits, SPACE, 
# 2 digits, colon, 2 digits, colon, 2 digits.
pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'

# 3. Find mismatches
matches = formatted_dates.str.match(pattern)
mismatches = formatted_dates[~matches]

print(f"Total records: {len(df)}")
print(f"Records matching format: {matches.sum()}")
print(f"Records failing format: {len(mismatches)}")

if len(mismatches) > 0:
    print("\nFirst 5 problematic values (after str() conversion):")
    print(mismatches.head().values)

Total records: 49416
Records matching format: 49416
Records failing format: 0


In [14]:
import re
import dataclasses

# 1. Ensure your DataFrame is prepared (crucial for avoiding NaN errors)
df = df.fillna(0)

results = []
# Regex for 'yyyy-MM-dd HH:mm:ss'
pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'
total_rows = len(df)

print(f"Starting check for {total_rows} rows...")

for index, row in df.iterrows():
    try:
        # Step A: Create the Ride object (Producer side)
        original_ride = ride_from_row(row)
        original_val = original_ride.lpep_pickup_datetime
        
        # Step B: Round-trip Serialization
        serialized = ride_serializer(original_ride)
        deserialized = ride_deserializer(serialized)
        final_val = deserialized.lpep_pickup_datetime
        
        # Step C: Validation Logic
        is_equal = (original_val == final_val)
        matches_format = bool(re.match(pattern, final_val))
        
        if not (is_equal and matches_format):
            results.append({
                'index': index,
                'original': original_val,
                'final': final_val,
                'is_equal': is_equal,
                'matches_format': matches_format
            })
            
    except Exception as e:
        results.append({'index': index, 'error': str(e)})

# 2. Display Results
print(f"Check complete.")
print(f"Total rows processed: {total_rows}")
print(f"Total failures found: {len(results)}")

if len(results) > 0:
    print("\nDetailed Failures (first 5):")
    for res in results[:5]:
        print(res)
else:
    print("\n✅ Success! All rows maintained perfect integrity through the serialization cycle.")

Starting check for 49416 rows...
Check complete.
Total rows processed: 49416
Total failures found: 0

✅ Success! All rows maintained perfect integrity through the serialization cycle.
